In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [6]:
import tensorflow as tf
from tensorflow import keras

from src.dataset import (
    create_datasets,
    compute_class_weights,
    get_data_augmentation,
)

from src.model import build_model

In [4]:
train_ds, val_ds, test_ds = create_datasets()

augmentation = get_data_augmentation()

weights = compute_class_weights(train_ds)

model = build_model(augmentation)

Found 28709 files belonging to 7 classes.
Using 22968 files for training.
Found 28709 files belonging to 7 classes.
Using 5741 files for validation.
Found 7178 files belonging to 7 classes.


In [7]:
model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)

In [8]:
checkpoint = keras.callbacks.ModelCheckpoint(
    filepath="../models/best_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

In [9]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    class_weight=weights,
    callbacks=[
        checkpoint,
        early_stopping,
        reduce_lr
    ]
)

Epoch 1/50
718/718 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - accuracy: 0.1632 - loss: 1.9840
Epoch 1: val_accuracy improved from None to 0.22505, saving model to ../models/best_model.keras

Epoch 1: finished saving model to ../models/best_model.keras
718/718 ━━━━━━━━━━━━━━━━━━━━ 74s 99ms/step - accuracy: 0.1682 - loss: 1.9427 - val_accuracy: 0.2250 - val_loss: 1.9062 - learning_rate: 0.0010
Epoch 2/50
718/718 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - accuracy: 0.1925 - loss: 1.8964
Epoch 2: val_accuracy did not improve from 0.22505
718/718 ━━━━━━━━━━━━━━━━━━━━ 72s 100ms/step - accuracy: 0.2019 - loss: 1.8806 - val_accuracy: 0.2068 - val_loss: 2.2717 - learning_rate: 0.0010
Epoch 3/50
718/718 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.2477 - loss: 1.8185
Epoch 3: val_accuracy improved from 0.22505 to 0.25240, saving model to ../models/best_model.keras

Epoch 3: finished saving model to ../models/best_model.keras
718/718 ━━━━━━━━━━━━━━━━━━━━ 71s 99ms/step - accuracy: 0.2520 - loss: 1.8066 - val